# 02 - Translate English to Tamil

This notebook runs the IndicTrans2 translation pipeline to create Tamil versions of the English airline tweets.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install -q torch transformers sentencepiece pandas scikit-learn pyyaml tqdm
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 13.3 MB/s eta 0:00:00


In [3]:
import os
os.chdir('/content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil')
print('Working directory:', os.getcwd())

Working directory: /content/drive/MyDrive/Airline_BERT_Multilingual/En_Tamil


In [4]:
!pip install -q \
    "transformers>=4.43,<4.47"\
    sentencepiece \
    sacremoses \
    accelerate \
    datasets \
    huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 53.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [5]:
# Step 1: Preprocess
!python scripts/preprocess.py

Loading dataset...
Loaded 14640 rows with columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']
Identified text column: 'text' (avg length: 103.8)
Identified sentiment column: 'airline_sentiment'

EXPLORATORY DATA ANALYSIS

Dataset shape: 14640 rows × 15 columns

--- Class Distribution ---
    negative:   9178 ( 62.7%)
     neutral:   3099 ( 21.2%)
    positive:   2363 ( 16.1%)

--- Missing Values ---
  Text column:      0 missing
  Sentiment column: 0 missing

--- Duplicates ---
  Duplicate tweets: 213

--- Tweet Length (characters) ---
  Min:        12
  Max:       186
  Mean:    103.8
  Median:    114
  Std:      36.3

--- Tweet Length (words) ---
  Min:         2
  Max:        36
  Mean:     17.7
  Median:     19


Applying conservative preprocessing...
  Dr

In [6]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [8]:
# Step 2: Translate to Tamil
# Use --limit for testing, remove for full translation
!python scripts/translate_to_tamil.py --batch-size 32

Loading English dataset...
Loaded 14427 English tweets
Tweets remaining to translate: 14427
Loading translation model: ai4bharat/indictrans2-en-indic-dist-200M
Using GPU: Tesla T4
2026-09-02 03:20:24.479545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
generation_config.json: 100% 163/163 [00:00<00:00, 877kB/s]
IndicTransToolkit processor loaded successfully

Translating 14427 tweets in 451 batches (batch_size=32)...
Translating:   0% 0/451 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the s

In [9]:
# Verify output
import pandas as pd
tamil_df = pd.read_csv('data/tamil_dataset.csv')
print(f'Tamil dataset: {len(tamil_df)} rows')
print(tamil_df.head())

Tamil dataset: 14427 rows
   original_tweet_id                                      original_text  \
0              11160  @USAirways I've been on hold for 40 minutes ju...   
1               1835  @united I assume that would be for the other 3...   
2              13540  @AmericanAir I need to know what to do? My fli...   
3              12239  @AmericanAir Thank you for the info! Changes n...   
4               5173  @SouthwestAir yes. Hung up and called a differ...   

                                                text sentiment language  \
0  @USAirways @USAirways ஒரு ரசீது பெறுவதற்காக நா...  negative       ta   
1  @united @united இது யுனைடெட் நிறுவனத்தால் வீழ்...  negative       ta   
2  @AmericanAir @AmericanAir நான் என்ன செய்வது என...  negative       ta   
3  @AmericanAir @AmericanAir தகவலுக்கு நன்றி! தொல...   neutral       ta   
4  @SouthwestAir @SouthwestAir ஆம். ஹங் அப் செய்த...  negative       ta   

                source translation_model  split  
0  machine_translation